# FrameWeave Hybrid v6.0 (V5.1) — HERBench **full** oracle eval (Qwen2.5-VL-7B)

Notebook **chỉ eval**. Không train. So sánh **Vanilla vs Hybrid**. Checkpoint v3/v4/v5 **không** load.

Video + parquet đọc từ Drive `/content/drive/MyDrive/HERBench_official` (pack đã extract, **không** snapshot tar). Oracle frames lấy từ `metadata_json["timestamps"]` lúc chạy — HERBench không phát hành thư mục frame.

**Pack gap (HF pin `30b42e97…`):** tar official thiếu `PersonPath22/uid_vid_00109.mp4` → skip **65** câu AGBI. Claim: **27 566 / 27 631**. Không dùng MEVA substitute.

```
outputs_reljoin_v6_qwen/
  checkpoints/best_external_val/Hybrid.pt
```

## Source code

Mở notebook này từ GitHub hoặc Colab. Run All clone [FrameWeave](https://github.com/nqMinh-GR/FrameWeave.git) vào `/content/FrameWeave`, rồi symlink `reljoin/` và `run_*.py` vào `/content`. Không upload file source.

Checkpoint và folder output train không nằm trong git. Lấy chúng từ `/content` hoặc Drive.

## Hardware

Colab **A100 40 GB**. Disk trống ≥ ~25 GB (chỉ Qwen ~18 GB; video ở Drive). Resume CSV trên Drive — full ~27k×2 method chạy nhiều giờ/ngày.

`TSO_ONLY=False`: full (trừ pack gap). `True`: chỉ Temporal Shot Ordering trên full (n > 152).


## 1. Configuration


In [ ]:
# 1. Eval-only config — FrameWeave v6.0 (V5.1) / Qwen2.5-VL-7B / HERBench full
from __future__ import annotations

from pathlib import Path

assert Path("/content").exists(), "Notebook này chỉ chạy trên Google Colab (/content)."

ROOT = Path("/content")
SMOKE_FIRST = False          # lần đầu True; smoke OK rồi đặt False trước full (resume CSV)
RUN_FULL = True
TSO_ONLY = False            # True = chỉ Temporal Shot Ordering trên full
HERBENCH_CONFIG = "full"
SKIP_UNRESOLVED = True      # pack gap uid_vid_00109
METHOD_VERSION = "6.0.0"

VARIANT_TO_METHOD = {
    "Hybrid": "hybrid",
}
REQUIRED_CKPTS = ["Hybrid.pt"]
METHODS = "vanilla,hybrid"

EXTRACT_DIR_CANDIDATES = [
    ROOT / "outputs_reljoin_v6_qwen",
    ROOT / "outputs_reljoin_v6_qwen (2)",
    Path("/content/drive/MyDrive/outputs_reljoin_v6_qwen"),
    Path("/content/drive/MyDrive/reljoin_v6_qwen_outputs") / "outputs_reljoin_v6_qwen",
    Path("/content/drive/MyDrive/reljoin_v6_qwen_outputs"),
]

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
MODEL_REVISION = "cc594898137f460bfe9f0759e9844b3ce807cfb5"
HERBENCH_ID = "DanBenAmi/HERBench"
HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"
HERBENCH_BASE_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/model_wrappers/base_vlm.py"
HERBENCH_ACC_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/calculate_accuracy.py"

VENV = ROOT / "qwen_env"
PYTHON = VENV / "bin" / "python"
OUT_DIR = ROOT / "outputs_reljoin_v6_qwen"
CKPT_DIR = OUT_DIR / "checkpoints" / "best_external_val"
PRED_DIR = OUT_DIR / "predictions"
MODEL_DIR = ROOT / "checkpoints" / "qwen25vl_7b"
DRIVE_HERBENCH = Path("/content/drive/MyDrive/HERBench_official")
HERBENCH_ROOT = DRIVE_HERBENCH
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
EVAL_DIR = ROOT / "herbench_eval_pins"

print({
    "SMOKE_FIRST": SMOKE_FIRST,
    "RUN_FULL": RUN_FULL,
    "TSO_ONLY": TSO_ONLY,
    "HERBENCH_CONFIG": HERBENCH_CONFIG,
    "SKIP_UNRESOLVED": SKIP_UNRESOLVED,
    "METHODS": METHODS,
    "REQUIRED_CKPTS": REQUIRED_CKPTS,
    "EXTRACT_DIR_CANDIDATES": [str(p) for p in EXTRACT_DIR_CANDIDATES],
    "CKPT_DIR": str(CKPT_DIR),
    "MODEL_DIR": str(MODEL_DIR),
    "HERBENCH_ROOT": str(HERBENCH_ROOT),
    "method": "FrameWeave-Hybrid",
    "method_version": METHOD_VERSION,
    "backbone": MODEL_ID,
})



## 2. Environment (`qwen_env`)

Pins: `torch==2.4.0`, `transformers==4.51.3`, `qwen-vl-utils`. flash-attn is optional (SDPA works).


In [ ]:
# 2. Bootstrap isolated qwen_env (Python 3.11 + torch 2.4.0 + transformers 4.51.3)
from __future__ import annotations

import os
import subprocess
from pathlib import Path

assert Path("/content").exists(), "This notebook targets Google Colab (/content)."
ROOT = Path("/content")
VENV = ROOT / "qwen_env"
VENV_BIN = VENV / "bin"
PYTHON = VENV_BIN / "python"
PIP = VENV_BIN / "pip"
LOG = ROOT / "outputs_reljoin_v6_qwen" / "logs" / "cell2_env.log"
LOG.parent.mkdir(parents=True, exist_ok=True)
LOG.write_text("", encoding="utf-8")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def sh(cmd, check=True, env=None):
    line = "$ " + " ".join(map(str, cmd))
    print(line, flush=True)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
        proc = subprocess.run(
            list(map(str, cmd)),
            check=False,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=env,
        )
        f.write(proc.stdout or "")
        f.write(f"\nexit={proc.returncode}\n\n")
    print(proc.stdout or "", end="")
    if check and proc.returncode != 0:
        print("LOG:", LOG)
        print((proc.stdout or "")[-4000:])
        raise RuntimeError(f"command failed ({proc.returncode}). See {LOG}")
    return proc


sh(["apt-get", "update", "-qq"], check=False)
sh(["apt-get", "install", "-y", "-qq", "python3.11", "python3.11-venv", "python3.11-dev", "ffmpeg"], check=False)
py311 = Path("/usr/bin/python3.11")
assert py311.exists(), "Không tìm thấy python3.11 — kiểm tra apt install."

need_recreate = True
if PYTHON.exists():
    ver = subprocess.check_output([str(PYTHON), "-V"], text=True).strip()
    print("Existing venv:", ver)
    need_recreate = "3.11" not in ver
if need_recreate:
    if VENV.exists():
        sh(["rm", "-rf", str(VENV)])
    sh([str(py311), "-m", "venv", "--copies", str(VENV)])
if not PIP.exists():
    sh([str(PYTHON), "-m", "ensurepip", "--upgrade"], check=False)
if not PIP.exists():
    get_pip = Path("/tmp/get-pip.py")
    sh(["curl", "-sSL", "https://bootstrap.pypa.io/get-pip.py", "-o", str(get_pip)])
    sh([str(PYTHON), str(get_pip)])
sh([str(PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools<70", "wheel"])
print("Python in venv:", subprocess.check_output([str(PYTHON), "-V"], text=True).strip())
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
    "torch==2.4.0", "torchvision==0.19.0",
    "--index-url", "https://download.pytorch.org/whl/cu121",
])
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed",
    "transformers==4.51.3", "numpy==1.25.0", "pillow==10.3.0", "moviepy==1.0.3",
    "huggingface_hub", "hf-transfer", "datasets", "accelerate", "einops", "sentencepiece",
    "opencv-python-headless", "pandas", "scipy", "tqdm", "ninja", "packaging", "qwen-vl-utils",
])
print("Installing qwen-vl-utils; flash-attn is optional (SDPA is enough)...")
fa_env = os.environ.copy()
fa_env["MAX_JOBS"] = "4"
sh(
    [str(PYTHON), "-m", "pip", "install", "flash-attn==2.7.0.post2", "--no-build-isolation"],
    env=fa_env,
    check=False,
)
check = subprocess.check_output([
    str(PYTHON), "-c",
    "import torch, transformers, qwen_vl_utils; "
    "print(torch.__version__); print(transformers.__version__); print(qwen_vl_utils.__file__); "
    "print(torch.__file__); print(transformers.__file__); "
    "import importlib.util; print('flash_attn', bool(importlib.util.find_spec('flash_attn')))",
], text=True)
print(check)
for line in check.splitlines():
    if "/site-packages/" in line:
        assert "/content/qwen_env/" in line, f"Package leaked outside venv: {line}"
print("qwen_env ready:", PYTHON)
print("LOG:", LOG)


## 3. Drive + pin HERBench evaluators

Không tải lại evaluator nếu SHA đã khớp (tránh GitHub 429). Không clone DTR.


In [ ]:
# 3. Drive + pin HERBench evaluator (skip download if SHA matches)
from __future__ import annotations

import hashlib
import os
import time
import urllib.request
from pathlib import Path

ROOT = Path("/content")
VENV = ROOT / "qwen_env"
PYTHON = VENV / "bin" / "python"
HERBENCH_ROOT = Path("/content/drive/MyDrive/HERBench_official")
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "qwen25vl_7b"
OUT_DIR = ROOT / "outputs_reljoin_v6_qwen"
EVAL_DIR = ROOT / "herbench_eval_pins"
HERBENCH_BASE_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/model_wrappers/base_vlm.py"
HERBENCH_ACC_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/calculate_accuracy.py"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"
JSDELIVR_BASE = "https://cdn.jsdelivr.net/gh/DanBenAmi/HERBench@main/evaluation/model_wrappers/base_vlm.py"
JSDELIVR_ACC = "https://cdn.jsdelivr.net/gh/DanBenAmi/HERBench@main/evaluation/calculate_accuracy.py"

for p in [MODEL_DIR.parent, EVAL_DIR, OUT_DIR, OUT_DIR / "predictions"]:
    p.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    RELJOIN_PERSIST = Path("/content/drive/MyDrive/reljoin_v6_qwen_outputs")
except Exception as exc:
    print("Drive mount skipped:", exc)
    RELJOIN_PERSIST = OUT_DIR / "persist_local"
if RELJOIN_PERSIST.resolve() == OUT_DIR.resolve():
    RELJOIN_PERSIST = OUT_DIR / "persist_local"
    print("Persist dir was out_dir; using", RELJOIN_PERSIST)
RELJOIN_PERSIST.mkdir(parents=True, exist_ok=True)
os.environ["RELJOIN_PERSIST"] = str(RELJOIN_PERSIST)
os.environ["RELJOIN_ROOT"] = str(ROOT)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def fetch_pinned(urls: list[str], dest: Path, expected: str) -> None:
    if dest.exists() and sha256_file(dest) == expected:
        print("Already pinned:", dest.name)
        return
    last_err = None
    for url in urls:
        for attempt in range(4):
            try:
                print(f"GET {url} (try {attempt + 1})")
                urllib.request.urlretrieve(url, dest)
                got = sha256_file(dest)
                if got == expected:
                    print("OK", dest.name)
                    return
                last_err = RuntimeError(f"SHA mismatch for {dest.name}: {got}")
            except Exception as exc:
                last_err = exc
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Cannot pin {dest.name}: {last_err}")


fetch_pinned([HERBENCH_BASE_URL, JSDELIVR_BASE], EVAL_DIR / "base_vlm.py", HERBENCH_BASE_SHA256)
fetch_pinned([HERBENCH_ACC_URL, JSDELIVR_ACC], EVAL_DIR / "calculate_accuracy.py", HERBENCH_ACC_SHA256)

print({
    "RELJOIN_PERSIST": str(RELJOIN_PERSIST),
    "OUT_DIR": str(OUT_DIR),
    "MODEL_DIR": str(MODEL_DIR),
    "EVAL_DIR": str(EVAL_DIR),
})




## 4. Clone source and verify


In [ ]:
# Fetch FrameWeave source. Run All clones the repo; do not upload .py files.
from __future__ import annotations

import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nqMinh-GR/FrameWeave.git"
REPO_DIR = Path("/content/FrameWeave")
ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])


def link(src: Path, dest: Path) -> None:
    src = src.resolve()
    if dest.is_symlink():
        if dest.resolve() != src:
            dest.unlink()
            dest.symlink_to(src)
        return
    if dest.exists():
        raise RuntimeError(
            f"{dest} already exists and is not the repo link. Restart the runtime, then Run All."
        )
    dest.symlink_to(src)


CODE = REPO_DIR / "qwen"
link(CODE / "reljoin", ROOT / "reljoin")
for script in sorted(CODE.glob("run_*.py")):
    link(script, ROOT / script.name)
print("FrameWeave source linked from", CODE)


In [ ]:
# 4. Verify FrameWeave Hybrid sources (eval needs runner + Qwen package)
from pathlib import Path

ROOT = Path("/content")
required = [
    ROOT / "run_herbench_oracle_reljoin.py",
    ROOT / "run_reljoin.py",
    ROOT / "reljoin" / "__init__.py",
    ROOT / "reljoin" / "config.py",
    ROOT / "reljoin" / "train.py",
    ROOT / "reljoin" / "reljoin_modules.py",
    ROOT / "reljoin" / "qwen_hooks.py",
    ROOT / "reljoin" / "herbench_oracle.py",
    ROOT / "reljoin" / "stats.py",
    ROOT / "reljoin" / "diagnostics.py",
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, "Missing files — clone or symlink failed:\n" + "\n".join(missing)
print("All FrameWeave Hybrid eval sources present.")
for p in required:
    print(f"  OK {p} ({p.stat().st_size} bytes)")


## 5. Gắn checkpoint (`outputs_reljoin_v6_qwen/checkpoints`)

Cùng layout train lite joint. Bắt buộc `Hybrid.pt` (`VARIANT=Hybrid`, `TRAIN_STAGE=joint`). Checkpoint v3/v4/v5 **không** load vào v6.0.

Ưu tiên `checkpoints/` trong repo đã clone. Drive chỉ dùng khi file đó không có.


In [ ]:
# Prefer the cloned checkpoint. Drive is the fallback in the next cell.
from __future__ import annotations

from pathlib import Path

ROOT = Path("/content")
REPO_DIR = Path("/content/FrameWeave")
PUBLISHED = REPO_DIR / "checkpoints" / "qwen"
OUT = ROOT / "outputs_reljoin_v6_qwen" / "checkpoints"

def link_published(src: Path, dest: Path) -> None:
    if not src.is_file():
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    src = src.resolve()
    if dest.is_symlink():
        if dest.resolve() != src:
            dest.unlink()
            dest.symlink_to(src)
        return
    if dest.exists():
        return
    dest.symlink_to(src)

linked = []
for bucket in ("best_external_val", "last"):
    src = PUBLISHED / bucket / "Hybrid.pt"
    dest = OUT / bucket / "Hybrid.pt"
    link_published(src, dest)
    if dest.is_file():
        linked.append(str(dest))
print("published checkpoints", linked or ["none — next cell may use Drive"])


In [ ]:
# 5. Bind extract checkpoints — same layout as train.py / make_output_tree
from __future__ import annotations

import json
import shutil
import zipfile
from pathlib import Path

ROOT = Path("/content")
OUT_DIR = ROOT / "outputs_reljoin_v6_qwen"
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "predictions").mkdir(parents=True, exist_ok=True)

try:
    REQUIRED_CKPTS
    EXTRACT_DIR_CANDIDATES
    VARIANT_TO_METHOD
except NameError:
    REQUIRED_CKPTS = ["Hybrid.pt"]
    VARIANT_TO_METHOD = {
        "Hybrid": "hybrid",
    }
    EXTRACT_DIR_CANDIDATES = [
        ROOT / "outputs_reljoin_v6_qwen",
        ROOT / "outputs_reljoin_v6_qwen (2)",
        Path("/content/drive/MyDrive/outputs_reljoin_v6_qwen"),
        Path("/content/drive/MyDrive/reljoin_v6_qwen_outputs") / "outputs_reljoin_v6_qwen",
        Path("/content/drive/MyDrive/reljoin_v6_qwen_outputs"),
    ]

v2_pt = ROOT / "outputs_reljoin_residual" / "checkpoints" / "best_external_val" / "RelJoin.pt"
if v2_pt.is_file():
    print("NOTE: found v2 RelJoin.pt at", v2_pt, "— không dùng cho v6.0")


def ckpt_complete(best_dir: Path) -> bool:
    return best_dir.is_dir() and all((best_dir / n).is_file() for n in REQUIRED_CKPTS)


def find_extract_root() -> Path:
    for cand in EXTRACT_DIR_CANDIDATES:
        best = cand / "checkpoints" / "best_external_val"
        if ckpt_complete(best):
            return cand
        last = cand / "checkpoints" / "last"
        if ckpt_complete(last):
            print("WARN: using checkpoints/last (best_external_val incomplete) at", cand)
            return cand
        if (cand / "Hybrid.pt").is_file():
            return cand
    zip_roots = [
        ROOT,
        Path("/content/drive/MyDrive"),
        Path("/content/drive/MyDrive/reljoin_v6_qwen_outputs"),
    ]
    for base in zip_roots:
        if not base.exists():
            continue
        for zpath in list(base.glob("*reljoin*.zip")) + list(base.glob("*output*.zip")):
            if not zpath.is_file():
                continue
            dest = Path("/tmp/vast_outputs_reljoin_v6_qwen")
            if dest.exists():
                shutil.rmtree(dest)
            print("Extracting", zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(dest)
            for cand in [dest, *dest.iterdir()]:
                if cand.is_dir() and ckpt_complete(cand / "checkpoints" / "best_external_val"):
                    return cand
                if cand.is_dir() and ckpt_complete(cand / "checkpoints" / "last"):
                    print("WARN: zip extract using checkpoints/last at", cand)
                    return cand
    scanned = "\n".join(
        f"  {c} exists={c.exists()} best={list((c / 'checkpoints' / 'best_external_val').glob('*.pt')) if (c / 'checkpoints' / 'best_external_val').exists() else 'NO DIR'}"
        for c in EXTRACT_DIR_CANDIDATES
    )
    raise FileNotFoundError(
        "Không thấy Hybrid.pt đúng layout train:\n"
        "  {extract}/checkpoints/best_external_val/Hybrid.pt\n"
        "Upload cả folder `outputs_reljoin_v6_qwen` (kèm checkpoints/) lên /content hoặc Drive.\n"
        f"Đã quét:\n{scanned}"
    )


EXTRACT_DIR = find_extract_root()
BEST_DIR = EXTRACT_DIR / "checkpoints" / "best_external_val"
LAST_DIR = EXTRACT_DIR / "checkpoints" / "last"
if ckpt_complete(BEST_DIR):
    CKPT_DIR = BEST_DIR
elif ckpt_complete(LAST_DIR):
    CKPT_DIR = LAST_DIR
else:
    reljoin_loose = EXTRACT_DIR / "Hybrid.pt"
    CKPT_DIR = EXTRACT_DIR if reljoin_loose.is_file() else BEST_DIR
CKPT_DIR.mkdir(parents=True, exist_ok=True)

resolved = {}
for name in REQUIRED_CKPTS:
    src = CKPT_DIR / name
    if not src.is_file():
        alt = LAST_DIR / name
        if alt.is_file():
            print(f"best missing {name} → copy from last/")
            shutil.copy2(alt, CKPT_DIR / name)
            src = CKPT_DIR / name
        else:
            resolved[name] = None
            continue
    mb = src.stat().st_size / 1e6
    print(f"OK {name}: {src} ({mb:.1f} MB)")
    resolved[name] = src

missing = [n for n, p in resolved.items() if p is None]
assert not missing, f"Thiếu checkpoint: {missing} dưới {CKPT_DIR} và {LAST_DIR}"

if CKPT_DIR.resolve() != LAST_DIR.resolve():
    for variant in VARIANT_TO_METHOD:
        name = f"{variant}.pt"
        if not (CKPT_DIR / name).is_file() and (LAST_DIR / name).is_file():
            print(f"missing {name} → copy from last/")
            shutil.copy2(LAST_DIR / name, CKPT_DIR / name)

found_methods = ["vanilla"]
for variant, method in VARIANT_TO_METHOD.items():
    path = CKPT_DIR / f"{variant}.pt"
    if path.is_file():
        if path.name not in REQUIRED_CKPTS:
            mb = path.stat().st_size / 1e6
            print(f"OK {path.name}: {path} ({mb:.1f} MB)")
        found_methods.append(method)
METHODS = ",".join(found_methods)

for aux in ["train_summary.json", "config.json", "environment.txt"]:
    src = EXTRACT_DIR / aux
    if src.exists() and src.resolve() != (OUT_DIR / aux).resolve():
        shutil.copy2(src, OUT_DIR / aux)
        print("Copied", aux)
log_src = EXTRACT_DIR / "logs" / "train_metrics.csv"
if log_src.exists():
    (OUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
    dst = OUT_DIR / "logs" / "train_metrics.csv"
    if log_src.resolve() != dst.resolve():
        shutil.copy2(log_src, dst)

summary_path = EXTRACT_DIR / "train_summary.json"
if summary_path.exists():
    print("train_summary:", json.loads(summary_path.read_text()))

print({
    "EXTRACT_DIR": str(EXTRACT_DIR),
    "CKPT_DIR": str(CKPT_DIR),
    "METHODS": METHODS,
    "pts": sorted(p.name for p in CKPT_DIR.glob("*.pt")),
})


## 6. Tải Qwen2.5-VL-7B + bind HERBench **full** trên Drive

PerceptionTest **không** tải. Video **không** snapshot từ HF. Cell bind `MyDrive/HERBench_official` và tải backbone (~18 GB) lên `/content/checkpoints/qwen25vl_7b`.


In [ ]:
# 6. Download Qwen2.5-VL-7B + bind Drive HERBench full (no tar)
from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "qwen_env" / "bin" / "python"
assert PYTHON.exists(), "Chạy cell 2 trước"
os.environ["PATH"] = f"{ROOT / 'qwen_env' / 'bin'}:{os.environ.get('PATH', '')}"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

HERBENCH_ROOT = Path("/content/drive/MyDrive/HERBench_official")
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "qwen25vl_7b"
MODEL_HF_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
MODEL_REVISION = "cc594898137f460bfe9f0759e9844b3ce807cfb5"
MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)

assert HERBENCH_ROOT.is_dir(), f"Chưa thấy {HERBENCH_ROOT} — mount đúng account Drive."
assert HERBENCH_VIDEOS.is_dir(), f"Chưa thấy {HERBENCH_VIDEOS}"
pq = HERBENCH_ROOT / "data" / "herbench_full.parquet"
assert pq.is_file(), f"Thiếu {pq}"

print("Downloading Qwen2.5-VL-7B (~18 GB bf16)...")
subprocess.run([str(PYTHON), "-c", f"""
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='{MODEL_HF_ID}',
    revision='{MODEL_REVISION}',
    local_dir=r'{MODEL_DIR}',
    ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
)
print('model OK')
"""], check=True)
assert (MODEL_DIR / "config.json").exists(), f"Missing {MODEL_DIR / 'config.json'}"

def count_mp4_here(folder: Path) -> int:
    if not folder.is_dir():
        return 0
    return sum(1 for x in os.listdir(folder) if x.lower().endswith(".mp4"))

top = sorted(os.listdir(HERBENCH_VIDEOS))
print("videos top", top)
n_pp = count_mp4_here(HERBENCH_VIDEOS / "PersonPath22")
n_tr = count_mp4_here(HERBENCH_VIDEOS / "trailers")
n_wt = count_mp4_here(HERBENCH_VIDEOS / "WildTrack")
epic = HERBENCH_VIDEOS / "HD_EPIC"
n_epic = 0
epic_subs = []
if epic.is_dir():
    for sub in sorted(os.listdir(epic)):
        d = epic / sub
        if d.is_dir():
            n = count_mp4_here(d)
            n_epic += n
            epic_subs.append((sub, n))
n_listed = n_pp + n_tr + n_wt + n_epic
has_00109 = (HERBENCH_VIDEOS / "PersonPath22" / "uid_vid_00109.mp4").is_file()
probe = HERBENCH_VIDEOS / "HD_EPIC" / "P01" / "P01-20240202-110250.mp4"
print({
    "PersonPath22": n_pp,
    "trailers": n_tr,
    "WildTrack": n_wt,
    "HD_EPIC": n_epic,
    "HD_EPIC_subs": epic_subs[:12],
    "n_mp4_listed": n_listed,
    "has_00109": has_00109,
    "probe_HD_EPIC": str(probe),
    "probe_ok": probe.is_file(),
    "parquet": str(pq),
    "model_config": True,
    "free_GiB": round(shutil.disk_usage(ROOT).free / 2**30, 2),
})
assert n_pp >= 30, f"PersonPath22 quá ít: {n_pp}"
assert n_tr >= 80, f"trailers quá ít: {n_tr}"
assert n_wt >= 50, f"WildTrack quá ít: {n_wt}"
assert n_epic >= 100, f"HD_EPIC nested quá ít: {n_epic} — Drive FUSE chưa list P0X?"
assert probe.is_file(), f"Thiếu probe {probe}"
print("Drive HERBench full bind OK (expect ~334 mp4, 00109 absent from official tar)")



## 7. Smoke (2 câu) rồi eval full (27 566 câu)

Smoke **phải** in `Loaded FrameWeave v6 ckpt .../Hybrid.pt`.

`--herbench-config full --skip-unresolved`. CSV fingerprint khác lite_v2. Persist: `MyDrive/reljoin_v6_qwen_outputs/predictions`.

Sau smoke: `SMOKE_FIRST = False` rồi chạy lại cell 7. Resume từ CSV. `TSO_ONLY=False` = full trừ pack gap.

Video đã chấm hết thì bỏ qua, không mở mp4. Video chưa xong được copy từng file xuống `/content/herbench_stage`, chấm xong thì xóa. Mỗi câu ghi ngay lên Drive CSV.


In [ ]:
# 7. HERBench oracle eval — smoke then full (NO training, NO DTR)
from __future__ import annotations

import json
import os
import shlex
import shutil
import signal
import subprocess
import threading
import time
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "qwen_env" / "bin" / "python"
RUNNER = ROOT / "run_herbench_oracle_reljoin.py"
VENV = ROOT / "qwen_env"
HERBENCH_ROOT = Path("/content/drive/MyDrive/HERBench_official")
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "qwen25vl_7b"
OUT_DIR = ROOT / "outputs_reljoin_v6_qwen"
PRED_DIR = OUT_DIR / "predictions"
EVAL_DIR = ROOT / "herbench_eval_pins"
CKPT = CKPT_DIR if "CKPT_DIR" in dir() else (OUT_DIR / "checkpoints" / "best_external_val")
PRED_DIR.mkdir(parents=True, exist_ok=True)
JOB_FILE = ROOT / "eval_reljoin.job.json"
DRIVE_DIR = Path("/content/drive/MyDrive/reljoin_v6_qwen_outputs")
WORKER_PERSIST = DRIVE_DIR / "predictions"
WORKER_PERSIST.mkdir(parents=True, exist_ok=True)

def _from_cell1(name: str, default):
    """Read config from the Colab/IPython user namespace (cell 1), not globals()."""
    try:
        ip = get_ipython()
    except NameError:
        ip = None
    ns = ip.user_ns if ip is not None else {}
    if name in ns:
        return ns[name]
    return default

SMOKE_FIRST = bool(_from_cell1("SMOKE_FIRST", True))
RUN_FULL = bool(_from_cell1("RUN_FULL", True))
METHODS = str(_from_cell1("METHODS", "vanilla,hybrid") or "vanilla,hybrid")
TSO_ONLY = bool(_from_cell1("TSO_ONLY", False))
HERBENCH_CONFIG = str(_from_cell1("HERBENCH_CONFIG", "full") or "full")
SKIP_UNRESOLVED = bool(_from_cell1("SKIP_UNRESOLVED", True))

HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
MODEL_REVISION = "cc594898137f460bfe9f0759e9844b3ce807cfb5"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"

needed = list(REQUIRED_CKPTS) if "REQUIRED_CKPTS" in dir() else ["Hybrid.pt"]
missing = [n for n in needed if not (CKPT / n).exists()]
assert not missing, (
    f"Chạy cell 5 trước. Thiếu {missing} trong {CKPT}. "
    "Upload folder `outputs_reljoin_v6_qwen/checkpoints/best_external_val/` lên /content."
)
assert (MODEL_DIR / "config.json").exists(), (
    f"Chạy cell 6 trước — missing {MODEL_DIR / 'config.json'} (model=false)."
)
assert PYTHON.exists(), "Chạy cell 2 trước — qwen_env missing."
assert RUNNER.is_file(), f"Missing {RUNNER}"

hy_pt = CKPT / "Hybrid.pt"
probe = HERBENCH_VIDEOS / "HD_EPIC" / "P01" / "P01-20240202-110250.mp4"
print({
    "Hybrid.pt": str(hy_pt),
    "Hybrid_MB": round(hy_pt.stat().st_size / 1e6, 1) if hy_pt.is_file() else None,
    "model_config": True,
    "MODEL_DIR": str(MODEL_DIR),
    "HERBENCH_ROOT": str(HERBENCH_ROOT),
    "probe_ok": probe.is_file(),
    "HERBENCH_CONFIG": HERBENCH_CONFIG,
    "SKIP_UNRESOLVED": SKIP_UNRESOLVED,
    "worker_persist": str(WORKER_PERSIST),
    "drive_dir": str(DRIVE_DIR) if DRIVE_DIR.exists() else None,
})
assert HERBENCH_VIDEOS.is_dir(), f"Chạy cell 6 — missing {HERBENCH_VIDEOS}"
assert probe.is_file(), f"Chạy cell 6 — missing probe {probe}"

base_cmd = [
    str(PYTHON), "-u", str(RUNNER),
    "--root", "/content",
    "--herbench-root", str(HERBENCH_ROOT),
    "--videos-root", str(HERBENCH_VIDEOS),
    "--model-dir", str(MODEL_DIR),
    "--out-dir", str(PRED_DIR),
    "--persist-dir", str(WORKER_PERSIST),
    "--stage-video-dir", "/content/herbench_stage",
    "--herbench-revision", HERBENCH_REVISION,
    "--model-revision", MODEL_REVISION,
    "--herbench-base-vlm", str(EVAL_DIR / "base_vlm.py"),
    "--herbench-accuracy", str(EVAL_DIR / "calculate_accuracy.py"),
    "--herbench-base-sha256", HERBENCH_BASE_SHA256,
    "--herbench-acc-sha256", HERBENCH_ACC_SHA256,
    "--methods", METHODS,
    "--reljoin-ckpt", str(CKPT),
    "--herbench-config", HERBENCH_CONFIG,
]
if TSO_ONLY:
    base_cmd.append("--tso-only")
if SKIP_UNRESOLVED:
    base_cmd.append("--skip-unresolved")
print({"TSO_ONLY": TSO_ONLY, "METHODS": METHODS, "HERBENCH_CONFIG": HERBENCH_CONFIG, "SKIP_UNRESOLVED": SKIP_UNRESOLVED})


def eval_env() -> dict:
    env = os.environ.copy()
    env.pop("PYTHONPATH", None)
    env.pop("PYTHONHOME", None)
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONNOUSERSITE"] = "1"
    env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    return env


def best_effort_copy(src: Path, dst: Path, timeout: float = 8.0) -> bool:
    src, dst = Path(src), Path(dst)
    if not src.exists():
        return False
    try:
        if dst.exists() and src.resolve() == dst.resolve():
            return True
    except OSError:
        return False
    try:
        dst.parent.mkdir(parents=True, exist_ok=True)
    except OSError as exc:
        print(f"WARN: mkdir {dst.parent}: {exc!r}", flush=True)
        return False
    err: list = []

    def _copy() -> None:
        try:
            tmp = dst.with_name(dst.name + ".tmp")
            shutil.copy2(src, tmp)
            tmp.replace(dst)
        except BaseException as exc:
            err.append(exc)

    t = threading.Thread(target=_copy, daemon=True)
    t.start()
    t.join(timeout)
    if t.is_alive():
        print(f"WARN: copy timed out {timeout}s: {src} -> {dst}", flush=True)
        return False
    if err:
        print(f"WARN: copy failed: {err[0]!r}", flush=True)
        return False
    return True


def pull_drive_csvs() -> None:
    src_dir = WORKER_PERSIST if WORKER_PERSIST.is_dir() else DRIVE_DIR
    if not src_dir.is_dir():
        return
    n = 0
    for src in list(src_dir.glob("predictions_reljoin_*.csv")) + list(src_dir.glob("summary_reljoin_*.json")):
        if best_effort_copy(src, PRED_DIR / src.name):
            n += 1
    if n:
        print(f"Pulled {n} artifact(s) from {src_dir} -> {PRED_DIR}", flush=True)


def push_drive_csvs() -> None:
    dest = WORKER_PERSIST if "WORKER_PERSIST" in dir() else DRIVE_DIR
    dest.mkdir(parents=True, exist_ok=True)
    n = 0
    for src in list(PRED_DIR.glob("predictions_reljoin_*.csv")) + list(PRED_DIR.glob("summary_reljoin_*.json")):
        if best_effort_copy(src, dest / src.name):
            n += 1
    if n:
        print(f"Pushed {n} artifact(s) -> {dest}", flush=True)
    else:
        print(f"Drive push skipped or timed out — local CSV remains in {PRED_DIR}", flush=True)


def _pid_alive(pid: int) -> bool:
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False


def _read_job() -> dict | None:
    if not JOB_FILE.exists():
        return None
    try:
        job = json.loads(JOB_FILE.read_text(encoding="utf-8"))
        pid = int(job["pid"])
    except Exception:
        return None
    if not _pid_alive(pid):
        JOB_FILE.unlink(missing_ok=True)
        return None
    return job


def _tail_log(log_path: Path, last_size: int) -> int:
    try:
        data = log_path.read_bytes()
    except OSError:
        return last_size
    if len(data) > last_size:
        print(data[last_size:].decode("utf-8", errors="replace"), end="", flush=True)
        return len(data)
    return last_size


def _kill_pg(pid: int) -> None:
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(pid, sig)
        except OSError:
            try:
                os.kill(pid, sig)
            except OSError:
                return
        if sig == signal.SIGTERM:
            time.sleep(2)


def _log_has_done(log_path: Path, label: str) -> bool:
    mark = "SMOKE OK" if label == "SMOKE" else "FULL OK"
    try:
        return mark in log_path.read_text(encoding="utf-8", errors="replace")
    except OSError:
        return False


def _wait_job(job: dict) -> bool:
    """True if worker exited. False if this cell was interrupted (worker still runs)."""
    pid = int(job["pid"])
    log_path = Path(job["log"])
    exit_path = Path(job["exit"])
    label = str(job.get("label") or "")
    last_size = 0
    silent_s = 0
    print(f"Waiting pid={pid} label={label} log={log_path}", flush=True)
    try:
        while True:
            prev = last_size
            last_size = _tail_log(log_path, last_size)
            if last_size == prev:
                silent_s += 15
                print(
                    f"[{time.strftime('%H:%M:%S')}] still running pid={pid} "
                    f"log={last_size}B silent={silent_s}s",
                    flush=True,
                )
            else:
                silent_s = 0
            if _log_has_done(log_path, label) and (not _pid_alive(pid) or silent_s >= 45):
                if _pid_alive(pid):
                    print(
                        f"{label} log already done; killing pid={pid} (CUDA exit hang).",
                        flush=True,
                    )
                    _kill_pg(pid)
                    time.sleep(1)
                if not exit_path.exists():
                    exit_path.write_text("0\n", encoding="utf-8")
                break
            if not _pid_alive(pid):
                time.sleep(1.0)
                last_size = _tail_log(log_path, last_size)
                break
            if silent_s >= 720:
                print(
                    "WARN: log silent >= 12 min (not a done marker). "
                    "Worker may be stuck; local CSV is still valid.",
                    flush=True,
                )
                silent_s = 0
            time.sleep(15)
    except KeyboardInterrupt:
        print(
            f"\nCell interrupted — {label} vẫn chạy pid={pid}. "
            f"Chạy lại cell này để reattach. Log: {log_path}",
            flush=True,
        )
        return False
    last_size = _tail_log(log_path, last_size)
    JOB_FILE.unlink(missing_ok=True)
    code = 1
    if exit_path.exists():
        try:
            code = int(exit_path.read_text(encoding="utf-8").strip() or "1")
        except ValueError:
            code = 1
    elif _log_has_done(log_path, label):
        code = 0
    if code != 0:
        raise RuntimeError(
            f"{label} exited {code}. "
            f"Xem {log_path}. "
            "venv torch / missing videos / state_dict / WARNING: no ckpt / missing Qwen config.json. "
            f"videos={HERBENCH_VIDEOS}."
        )
    return True


def run_eval(cmd: list[str], label: str) -> bool:
    live = _read_job()
    if live is not None:
        print(
            f"Job đang chạy ({live.get('label')} pid={live['pid']}) — reattach, không start mới.",
            flush=True,
        )
        return _wait_job(live)

    log_path = ROOT / f"eval_reljoin_{label.lower()}.log"
    exit_path = ROOT / f"eval_reljoin_{label.lower()}.exit"
    exit_path.unlink(missing_ok=True)
    print("$", " ".join(cmd), flush=True)
    print("detached log:", log_path, flush=True)
    inner = " ".join(shlex.quote(c) for c in cmd)
    script = (
        f"{inner} > {shlex.quote(str(log_path))} 2>&1; "
        f"echo $? > {shlex.quote(str(exit_path))}"
    )
    proc = subprocess.Popen(
        ["bash", "-c", script],
        cwd="/content",
        env=eval_env(),
        start_new_session=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    job = {
        "pid": proc.pid,
        "label": label,
        "log": str(log_path),
        "exit": str(exit_path),
    }
    JOB_FILE.write_text(json.dumps(job), encoding="utf-8")
    print(f"started pid={proc.pid} (Stop cell không giết job)", flush=True)
    return _wait_job(job)


def _after_success(label: str) -> None:
    push_drive_csvs()
    if label == "SMOKE":
        print("SMOKE xong — kiểm tra log phải có 'Loaded FrameWeave v6 ckpt'.", flush=True)
    elif label == "FULL":
        print("FULL eval xong. CSV/summary trong", PRED_DIR, flush=True)


pull_drive_csvs()

live = _read_job()
if live is not None:
    ok = _wait_job(live)
    if not ok:
        print("Re-run cell 7 để reattach (đừng start job thứ hai).")
    else:
        _after_success(str(live.get("label") or ""))
        if live.get("label") == "SMOKE" and RUN_FULL:
            if run_eval(list(base_cmd), "FULL"):
                _after_success("FULL")
            else:
                print("Full vẫn chạy nền. Re-run cell 7 để reattach.")
else:
    smoke_ok = True
    if SMOKE_FIRST:
        smoke_ok = run_eval(base_cmd + ["--smoke"], "SMOKE")
        if smoke_ok:
            _after_success("SMOKE")
        else:
            print("Smoke vẫn chạy nền. Re-run cell 7 để reattach (đừng start full song song).")
    if smoke_ok and RUN_FULL:
        if run_eval(list(base_cmd), "FULL"):
            _after_success("FULL")
        else:
            print("Full vẫn chạy nền. Re-run cell 7 để reattach.")
    elif not RUN_FULL:
        print("RUN_FULL=False — chỉ smoke.")




## 8. Kết quả Vanilla vs Hybrid

Không DTR. In accuracy paired Vanilla vs Hybrid từ `summary_reljoin_*.json`. TSO / non-TSO / task-level nằm trong official HERBench accuracy.


In [ ]:
# 8. Summarize FrameWeave Hybrid full eval (Vanilla vs Hybrid)
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

ROOT = Path("/content")
OUT = ROOT / "outputs_reljoin_v6_qwen"
PRED = OUT / "predictions"
OUT.mkdir(parents=True, exist_ok=True)

metrics = OUT / "logs" / "train_metrics.csv"
if not metrics.exists() and "EXTRACT_DIR" in dir():
    metrics = Path(EXTRACT_DIR) / "logs" / "train_metrics.csv"
if metrics.exists():
    print("Lite train_metrics (external PerceptionTest, not HERBench):")
    display(pd.read_csv(metrics))
else:
    print("No train_metrics.csv staged.")

rel_summaries = sorted(PRED.glob("summary_reljoin_*.json"), key=lambda p: p.stat().st_mtime) if PRED.exists() else []
sci = {
    "herbench_over_vanilla": {"ok": False, "notes": "PENDING paired CI"},
    "paired_ci_promising": {"ok": False, "notes": "PENDING statistical_tests.json"},
}
if rel_summaries:
    rel = json.loads(rel_summaries[-1].read_text())
    print("FrameWeave Hybrid HERBench summary:", rel_summaries[-1].name)
    if rel.get("tso_only"):
        print("Subset: Temporal Shot Ordering only")
    print("herbench_config", rel.get("herbench_config"), "n_resolved", rel.get("n_resolved"),
          "n_missing", rel.get("n_missing"), "pack_gap", rel.get("pack_gap_stems"))
    print(json.dumps(rel.get("summary", {}), indent=2)[:6000])
    stats = rel.get("stats") or {}
    paired_keys = [k for k in stats if str(k).startswith("vanilla_vs_")]
    for key in paired_keys:
        boot = (stats.get(key) or {}).get("bootstrap") or {}
        payload = {
            "ok": boot.get("ci_low", -1) > 0,
            "notes": f"{key} delta={boot.get('delta')} CI=({boot.get('ci_low')},{boot.get('ci_high')})",
        }
        sci[key] = payload
        if key == "vanilla_vs_hybrid":
            sci["herbench_over_vanilla"] = payload
            sci["paired_ci_promising"] = dict(payload)
        elif sci["herbench_over_vanilla"].get("notes") == "PENDING paired CI":
            sci["herbench_over_vanilla"] = payload
            sci["paired_ci_promising"] = dict(payload)
    rows = []
    for method, payload in (rel.get("summary") or {}).items():
        acc = payload.get("strict_accuracy", payload.get("accuracy"))
        if acc is None and isinstance(payload.get("official"), dict):
            acc = payload["official"].get("accuracy")
        n = payload.get("n") or payload.get("n_ok")
        rows.append({"method": method, "n": n, "accuracy": acc})
        by_task = payload.get("by_task_type") or {}
        if by_task:
            print(f"  {method} by_task_type:", {k: v.get("strict_accuracy") for k, v in by_task.items()})
    if rows:
        print("Method accuracy:")
        display(pd.DataFrame(rows))
    stats_path = PRED / "statistical_tests.json"
    if stats_path.exists():
        print("statistical_tests.json:")
        print(stats_path.read_text()[:4000])
else:
    print("Chưa có summary_reljoin_*.json — chạy cell 7 full trước.")

pairs = sorted(PRED.glob("predictions_reljoin_*.csv"), key=lambda p: p.stat().st_mtime) if PRED.exists() else []
if pairs:
    df = pd.read_csv(pairs[-1])
    ok = df[df["status"] != "runtime_error"] if "status" in df.columns else df
    print("Pred CSV:", pairs[-1].name, "rows", len(df))
    if "method" in ok.columns and "is_correct" in ok.columns:
        display(ok.groupby("method")["is_correct"].mean().rename("accuracy").to_frame())

report = {"scientific": sci, "pred_summaries": [str(p) for p in rel_summaries[-3:]]}
(OUT / "eval_full_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(sci, indent=2))



## 9. Copy artifact sang Drive


In [ ]:
# 9. Persist predictions/summaries to Drive
from __future__ import annotations

import os
import shutil
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path("/content")
PRED = ROOT / "outputs_reljoin_v6_qwen" / "predictions"
PERSIST = Path(os.environ.get("RELJOIN_PERSIST", "/content/drive/MyDrive/reljoin_v6_qwen_outputs"))
if not PERSIST.exists():
    print("Drive persist không có — bỏ qua copy.")
else:
    dest = PERSIST / "eval_herbench_full"
    try:
        same = dest.resolve() == PRED.resolve()
    except OSError:
        same = False
    if same:
        print("Persist dest is predictions dir; skip copy.")
    else:
        dest.mkdir(parents=True, exist_ok=True)
        n = 0
        if PRED.exists():
            for p in PRED.glob("*"):
                if not p.is_file():
                    continue
                try:
                    if p.resolve() == (dest / p.name).resolve():
                        continue
                except OSError:
                    pass
                shutil.copy2(p, dest / p.name)
                n += 1
        stamp = dest / "copied_at.txt"
        stamp.write_text(datetime.now(timezone.utc).isoformat(), encoding="utf-8")
        print(f"Copied {n} files → {dest}")



In [ ]:
# Zip HERBench eval results and download to your laptop
from __future__ import annotations

import zipfile
from datetime import datetime, timezone
from pathlib import Path

from google.colab import files

ROOT = Path("/content")
OUT = ROOT / "outputs_reljoin_v6_qwen"
PRED = OUT / "predictions"
DRIVE = Path("/content/drive/MyDrive/reljoin_v6_qwen_outputs") / "eval_full"

skip_dirs = {"oracle_clips", "checkpoints"}
skip_suffixes = {".pt", ".bin", ".safetensors"}

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
zip_path = ROOT / f"reljoin_v6_qwen_eval_{stamp}.zip"

n = 0
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    extra = [OUT / "eval_full_report.json"]
    if PRED.exists():
        for p in PRED.rglob("*"):
            if not p.is_file():
                continue
            if any(part in skip_dirs for part in p.relative_to(PRED).parts):
                continue
            if p.suffix.lower() in skip_suffixes:
                continue
            zf.write(p, arcname=f"predictions/{p.relative_to(PRED)}")
            n += 1
    for p in extra:
        if p.is_file():
            zf.write(p, arcname=p.name)
            n += 1
    if DRIVE.exists():
        for p in DRIVE.glob("*"):
            if p.is_file() and p.suffix.lower() not in skip_suffixes:
                zf.write(p, arcname=f"drive_eval_full/{p.name}")
                n += 1

print(f"Packed {n} files → {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
files.download(str(zip_path))
